# Практическая работа № 2
## Контрольное восстановление — универсальный Colab для вариантов 1–25

Этот ноутбук нужен только для **простой проверки резервного копирования и восстановления файла**.

### Что подготовлено преподавателем
Файл `grades_all_variants.csv` содержит:
- все **25 индивидуальных вариантов**;
- по **10 условных записей обучающихся** для каждого варианта;
- формулировки трёх заданий своего варианта;
- условные оценки без реальных персональных данных.

В файле специально есть лишние данные. Студенту нужен только **свой вариант**.

### Что нужно сделать студенту
1. Загрузить файл `grades_all_variants.csv`.
2. Ниже указать номер своего варианта от 1 до 25.
3. Запустить ячейки по порядку.
4. Ноутбук сам оставит только 10 строк вашего варианта и создаст файл `grades.csv`.
5. Затем будет создана резервная копия `backup_grades.csv`.
6. Из неё будет восстановлен файл `restored_grades.csv`.
7. В конце автоматически сравниваются SHA-256 и число строк.

**Важно:** вручную удалять остальные 24 варианта не нужно. Это делает код.

Для примера по умолчанию указан **вариант 25**. Если у вас другой вариант, замените только число.


In [ ]:
# Укажите только номер своего варианта: от 1 до 25.
MY_VARIANT = 2

if not 1 <= MY_VARIANT <= 25:
    raise ValueError("Номер варианта должен быть от 1 до 25.")

print("Выбран вариант:", MY_VARIANT)


Выбран вариант: 2


## Шаг 1. Загрузка универсального CSV

Нажмите кнопку **«Выбрать файлы»** и загрузите `grades_all_variants.csv`.


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError("Файл не загружен.")

SOURCE_FILE = next(iter(uploaded))
print("Загружен файл:", SOURCE_FILE)


Saving grades_all_variants.csv to grades_all_variants (1).csv
Загружен файл: grades_all_variants (1).csv


## Шаг 2. Оставляем только свой вариант

Код:
- читает универсальный CSV;
- выбирает строки только вашего варианта;
- показывает три индивидуальных задания;
- создаёт `grades.csv` с 10 условными записями.

Это и есть файл, который далее используется для контрольного восстановления.


In [ ]:
import csv

selected = []

with open(SOURCE_FILE, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames

    required = {
        "variant", "variant_name", "task_1", "task_2", "task_3",
        "student_code", "class_group", "subject", "grade", "comment"
    }

    if fieldnames is None or not required.issubset(set(fieldnames)):
        raise ValueError("Загружен не тот CSV-файл или изменена его структура.")

    for row in reader:
        if int(row["variant"]) == MY_VARIANT:
            selected.append(row)

if len(selected) != 10:
    raise ValueError(
        f"Для варианта {MY_VARIANT} найдено {len(selected)} строк вместо 10."
    )

first = selected[0]

print(f"Вариант {MY_VARIANT}. {first['variant_name']}")
print()
print("1.", first["task_1"])
print("2.", first["task_2"])
print("3.", first["task_3"])

work_fields = [
    "variant",
    "student_code",
    "class_group",
    "subject",
    "grade",
    "comment",
]

with open("grades.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=work_fields, lineterminator="\n")
    writer.writeheader()
    for row in selected:
        writer.writerow({key: row[key] for key in work_fields})

print()
print("Создан файл grades.csv: 10 условных записей + строка заголовка.")


Вариант 2. Педагогический сегмент

1. Добавьте риск распространения вредоносного ПО с ПК педагога.
2. Покажите доступ педагогов к LMS, журналу и файловому серверу.
3. Укажите, к каким внутренним ресурсам педагогический сегмент не должен иметь прямого доступа.

Создан файл grades.csv: 10 условных записей + строка заголовка.


## Шаг 3. Посмотрим содержимое `grades.csv`

В нём нет реальных ФИО: используются условные коды `U01`–`U10`.


In [ ]:
with open("grades.csv", "r", encoding="utf-8-sig") as f:
    print(f.read())


variant,student_code,class_group,subject,grade,comment
2,U01,8А,Информатика,5,работа выполнена
2,U02,8А,Информатика,4,работа выполнена
2,U03,8А,Информатика,5,работа выполнена
2,U04,8А,Информатика,3,нужно исправление
2,U05,8А,Информатика,4,работа выполнена
2,U06,8А,Информатика,5,работа выполнена
2,U07,8А,Информатика,4,работа выполнена
2,U08,8А,Информатика,3,нужно исправление
2,U09,8А,Информатика,5,работа выполнена
2,U10,8А,Информатика,4,работа выполнена



## Шаг 4. Резервная копия и восстановление

Моделируется простой учебный сценарий:

`grades.csv` → **резервная копия** → `backup_grades.csv`

`backup_grades.csv` → **восстановление** → `restored_grades.csv`

Для практической работы этого достаточно, чтобы проверить принцип: после восстановления файл должен быть **точно таким же**, как исходный.


In [ ]:
import shutil

shutil.copyfile("grades.csv", "backup_grades.csv")
shutil.copyfile("backup_grades.csv", "restored_grades.csv")

print("Резервная копия создана: backup_grades.csv")
print("Файл восстановлен: restored_grades.csv")


Резервная копия создана: backup_grades.csv
Файл восстановлен: restored_grades.csv


## Шаг 5. Проверяем результат

Используем две понятные проверки:

- **SHA-256** — цифровой «отпечаток» файла. Если отпечатки одинаковые, содержимое файлов совпадает.
- **Число строк** — дополнительная наглядная проверка. Должно быть 11 строк: заголовок + 10 записей.

Если обе проверки совпадают, контрольное восстановление выполнено успешно.


In [ ]:
import hashlib

def sha256_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def line_count(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return sum(1 for _ in f)

source_sha = sha256_file("grades.csv")
restored_sha = sha256_file("restored_grades.csv")

source_lines = line_count("grades.csv")
restored_lines = line_count("restored_grades.csv")

sha_ok = source_sha == restored_sha
lines_ok = source_lines == restored_lines
all_ok = sha_ok and lines_ok

print("SHA-256 исходного файла:")
print(source_sha)
print()
print("SHA-256 восстановленного файла:")
print(restored_sha)
print()
print("Число строк:", source_lines, "и", restored_lines)
print()

if all_ok:
    print("✅ Контрольное восстановление успешно.")
    print("Восстановленная копия идентична исходному файлу.")
else:
    print("❌ Проверка не пройдена. Файлы различаются.")


SHA-256 исходного файла:
1339270901336f5325bb840979c1789dbbfdfd526dff97931d8d41076acef787

SHA-256 восстановленного файла:
1339270901336f5325bb840979c1789dbbfdfd526dff97931d8d41076acef787

Число строк: 11 и 11

✅ Контрольное восстановление успешно.
Восстановленная копия идентична исходному файлу.


## Шаг 6. Готовая таблица для отчёта

Значения ниже можно перенести в **«Таблицу 8 — Результат контрольного восстановления»**.


In [ ]:
from IPython.display import display, Markdown

result_md = f"""
| Проверка | Исходный файл | Восстановленный файл |
|---|---|---|
| SHA-256 | `{source_sha}` | `{restored_sha}` |
| Число строк | {source_lines} | {restored_lines} |
| Итог | {'совпадает' if sha_ok else 'не совпадает'} | {'совпадает' if sha_ok else 'не совпадает'} |

**Вывод:** {'контрольные суммы SHA-256 и число строк совпадают; восстановленная копия идентична исходному файлу, резервное копирование работоспособно.' if all_ok else 'результаты не совпадают; необходимо проверить процедуру резервного копирования и восстановления.'}
"""
display(Markdown(result_md))



| Проверка | Исходный файл | Восстановленный файл |
|---|---|---|
| SHA-256 | `1339270901336f5325bb840979c1789dbbfdfd526dff97931d8d41076acef787` | `1339270901336f5325bb840979c1789dbbfdfd526dff97931d8d41076acef787` |
| Число строк | 11 | 11 |
| Итог | совпадает | совпадает |

**Вывод:** контрольные суммы SHA-256 и число строк совпадают; восстановленная копия идентична исходному файлу, резервное копирование работоспособно.


## Шаг 7. Что написать в отчёте

Можно использовать короткую формулировку:

> Для контрольного восстановления из универсального набора был выбран индивидуальный вариант.  
> Сформирован файл `grades.csv` с 10 условными записями без реальных персональных данных.  
> Создана резервная копия `backup_grades.csv`, после чего выполнено восстановление в `restored_grades.csv`.  
> Контрольные суммы SHA-256 и число строк исходного и восстановленного файлов совпали.  
> Следовательно, восстановленная копия идентична исходной, а учебная процедура резервного копирования и восстановления выполнена успешно.

### Важно
Индивидуальные задания варианта выполняются **в основном отчёте**. Этот Colab не оценивает правильность сетевой схемы, RBAC или выбранных мер защиты. Он нужен только для простой практической проверки резервного копирования и восстановления.


In [ ]:
# Необязательно: скачать рабочие файлы на компьютер.
from google.colab import files

# При необходимости снимите символ # перед нужной строкой:
# files.download("grades.csv")
# files.download("backup_grades.csv")
# files.download("restored_grades.csv")

print("Готово. При необходимости можно скачать три CSV-файла.")


Готово. При необходимости можно скачать три CSV-файла.
